In [1]:
!pip install scikit-learn xgboost

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from xgboost import XGBClassifier
#from sklearn.metrics.pairwise import cosine_similarity

In [3]:
BASE = "/kaggle/input/notebooks/krishnaiiith/02-feature-extraction"
BASE2 = "/kaggle/input/notebooks/krishnaiiith/01-dataset-preparation"

# embeddings
text_emb = np.load(BASE + "/text_emb.npy")
img_emb = np.load(BASE + "/img_emb.npy")

hash_emb = np.load(BASE + "/hash_emb.npy")

# hashtag stats
hashtag_count = np.load(BASE + "/hashtag_count.npy")
hashtag_freq = np.load(BASE + "/hashtag_freq.npy")

# semantic features from Notebook 2
semantic_features = np.load(BASE + "/semantic_features.npy")

# labels
df = pd.read_csv(BASE2 + "/labeled.csv")
y = df["label"].values

In [4]:
# Difference features between embeddings
text_img_diff = np.abs(text_emb - img_emb)
text_hash_diff = np.abs(text_emb - hash_emb)

In [5]:
def cosine_pairwise(a, b):

    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)

    return np.sum(a * b, axis=1, keepdims=True)

In [6]:
S_text_hash = cosine_pairwise(text_emb, hash_emb)
S_img_hash = cosine_pairwise(img_emb, hash_emb)
S_text_img = cosine_pairwise(text_emb, img_emb)

semantic_cross = np.concatenate([
    S_text_hash,
    S_img_hash,
    S_text_img
], axis=1)

print("Cross-modal semantic shape:", semantic_cross.shape)

Cross-modal semantic shape: (11902, 3)


In [7]:
count = hashtag_count.reshape(-1,1)
freq = hashtag_freq.reshape(-1,1)

X = np.concatenate([
    text_emb,
    img_emb,
    hash_emb,
    text_img_diff,
    text_hash_diff,
    count,
    freq,
    S_text_hash,
    S_img_hash,
    S_text_img
], axis=1)

print("Final feature matrix shape:", X.shape)

Final feature matrix shape: (11902, 2565)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [9]:
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

pos_weight = neg / pos

In [10]:
clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=pos_weight,
    random_state=42
)

In [11]:
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("FULL MULTIMODAL MODEL (TEXT + IMAGE + HASHTAG SEMANTICS)")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

FULL MULTIMODAL MODEL (TEXT + IMAGE + HASHTAG SEMANTICS)
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      2083
           1       0.52      0.37      0.43       298

    accuracy                           0.88      2381
   macro avg       0.71      0.66      0.68      2381
weighted avg       0.86      0.88      0.87      2381

ROC-AUC: 0.8213928993739671
